# Programe sua GPU com OpenMP

Autores:
_Hermes Senger_ e
_Jaime Freire de Souza_

Data de criação: 16/04/2022

Última modificação:

    15/07/2026 - Troca de FP64 por FP32 + otimizações SIMD
    24/04/2024 - Instalação do gcc13

## Configuração do ambiente

Precisaremos de um compilador capaz de gerar código executável para GPUs. O GCC 13 faz isso. Também precisaremos do CUDA, mas esse já está previamente instalado.

```
# This is formatted as code
```




In [1]:
%%shell
#ajustar repositorio de compiladores
true | add-apt-repository ppa:ubuntu-toolchain-r/test

PPA publishes dbgsym, you may need to include 'main/debug' component
Repository: 'deb https://ppa.launchpadcontent.net/ubuntu-toolchain-r/test/ubuntu/ jammy main'
Description:
Toolchain test builds; see https://wiki.ubuntu.com/ToolChain

More info: https://launchpad.net/~ubuntu-toolchain-r/+archive/ubuntu/test
Adding repository.
Adding deb entry to /etc/apt/sources.list.d/ubuntu-toolchain-r-ubuntu-test-jammy.list
Adding disabled deb-src entry to /etc/apt/sources.list.d/ubuntu-toolchain-r-ubuntu-test-jammy.list
Adding key to /etc/apt/trusted.gpg.d/ubuntu-toolchain-r-ubuntu-test.gpg with fingerprint C8EC952E2A0E1FBDC5090F6A2C277A0A352154E5
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.

In [2]:
%%shell
#instalar gcc 13 e primos
apt install gcc-13 g++-13 gcc-13-offload-nvptx libgomp1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-13 cpp-13-x86-64-linux-gnu g++-13-x86-64-linux-gnu gcc-13-base
  gcc-13-x86-64-linux-gnu gcc-16-base libasan8 libatomic1 libcc1-0
  libgcc-13-dev libgcc-s1 libgomp-plugin-nvptx1 libhwasan0 libitm1 liblsan0
  libquadmath0 libstdc++-13-dev libstdc++6 libtsan2 libubsan1 nvptx-tools
Suggested packages:
  gcc-13-locales cpp-13-doc g++-13-multilib gcc-13-doc gcc-13-multilib
  libcuda1 | libnvidia-tesla-cuda1 | libcuda1-any libstdc++-13-doc
  nvidia-cuda-toolkit
The following NEW packages will be installed:
  cpp-13 cpp-13-x86-64-linux-gnu g++-13 g++-13-x86-64-linux-gnu gcc-13
  gcc-13-base gcc-13-offload-nvptx gcc-13-x86-64-linux-gnu gcc-16-base
  libasan8 libgcc-13-dev libgomp-plugin-nvptx1 libhwasan0 libstdc++-13-dev
  libtsan2 nvptx-tools
The following packages will be upgraded:
  libatomic1 libcc1-0 libgcc-s1 libgomp1 libitm1 liblsan0

In [3]:
!nvcc --version
!ls -l /usr/bin

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
total 383056
-rwxr-xr-x 1 root root       51648 Feb  8  2024 '['
-rwxr-xr-x 1 root root          39 Aug 15  2020  7z
-rwxr-xr-x 1 root root          40 Aug 15  2020  7za
-rwxr-xr-x 1 root root          40 Aug 15  2020  7zr
lrwxrwxrwx 1 root root          25 Jun  4 13:21  aclocal -> /etc/alternatives/aclocal
-rwxr-xr-x 1 root root       36020 Mar 18  2022  aclocal-1.16
-rwxr-xr-x 1 root root       14488 May 11  2024  acyclic
-rwxr-xr-x 1 root root       14481 Jun  4 13:21  add-apt-repository
-rwxr-xr-x 1 root root       14720 Apr  9  2024  addpart
lrwxrwxrwx 1 root root          26 Feb  5  2025  addr2line -> x86_64-linux-gnu-addr2line
-rwxr-xr-x 1 root root        1887 Mar  4  2022  aggregate_profile
-rwxr-xr-x 1 root root       14576 Feb  9  2022  ambiguous_words
lrwxrwxrwx 1 root 

In [4]:
!ln -sfnv /usr/bin/gcc-13 /usr/bin/gcc
!which gcc
!ls -l /usr/bin/gcc
!gcc --version


'/usr/bin/gcc' -> '/usr/bin/gcc-13'
/usr/bin/gcc
lrwxrwxrwx 1 root root 15 Jul 15 16:57 /usr/bin/gcc -> /usr/bin/gcc-13
gcc (Ubuntu 13.4.0-6ubuntu1~22~ppa2) 13.4.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [5]:
!ls -l /usr/local

total 56
drwxr-xr-x 1 1000 1000 4096 Jun 29 13:34 bin
drwxr-xr-x 3 root root 4096 Jul 14 13:26 colab
lrwxrwxrwx 1 root root   22 Mar 10  2025 cuda -> /etc/alternatives/cuda
lrwxrwxrwx 1 root root   25 Mar 10  2025 cuda-12 -> /etc/alternatives/cuda-12
drwxr-xr-x 1 root root 4096 Mar 10  2025 cuda-12.8
drwxr-xr-x 1 1000 1000 4096 Jun 29 13:31 etc
drwxr-xr-x 2 root root 4096 Jan 26  2025 games
drwxr-xr-x 1 1000 1000 4096 Jun 29 13:31 include
drwxr-xr-x 1 1000 1000 4096 Jun 29 13:31 lib
drwxr-xr-x 3 1000 1000 4096 Apr  9 20:16 libexec
-rw-r--r-- 1 1000 1000 1321 Apr  9 20:16 LICENSE.md
lrwxrwxrwx 1 root root    9 Jan 26  2025 man -> share/man
drwxr-xr-x 3 root root 4096 Jun 29 13:31 opt
drwxr-xr-x 2 root root 4096 Jan 26  2025 sbin
drwxr-xr-x 1 1000 1000 4096 Jun 29 13:31 share
drwxr-xr-x 2 root root 4096 Jan 26  2025 src


In [6]:
%%shell
#ajustar link para cuda-10
#ln -sfnv /usr/local/cuda-10.0/ /usr/local/cuda
#instalar matplotlib
pip install -q matplotlib numpy


Vamos testar nosso ambiente de programação.

In [7]:
%%writefile test.c

#include <omp.h>
#include <stdio.h>

int main() {
  int numdevices = omp_get_num_devices();
  printf("number of devices= %d\n", numdevices);
}

Writing test.c


## Compilando

Para compilar um programa OpenMP para GPUs neste ambiente, use os flags abaixo.


In [8]:
%%shell

#clang -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda test.c -o teste
gcc -fno-lto -fopenmp -fstack-protector test.c -o test
./test

number of devices= 1


Vamos verificar o modelo de GPU alocada.

## Exemplo 1: Cálculo de Pi - versão serial

O programa a seguir calcula o valos re Pi pelo método de integração numérica. Esta primeira versão trabalha de forma serial, somente na CPU. Nas próximas versões, nós faremos melhorias nesse programa para acelerar o seu desempenho.


In [9]:
%%writefile ex1-pi_serial.c
// Versão serial em FP32.
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    start_time = omp_get_wtime();

    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;
    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing ex1-pi_serial.c


In [10]:
%%shell

#clang -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda ex1-pi_serial.c -o ex1-pi_serial
gcc -fno-lto -fopenmp  -fstack-protector ex1-pi_serial.c -o ex1-pi_serial
./ex1-pi_serial

pi = 0.67108864, 100000000 steps, 0.734403 secs


## Exemplo 2: Uma versão paralela pra CPU - Pi-V1.0.c

O próximo exemplo mostra uma forma de paralelizar o algoritmo para execução em CPU, utilizando os seguintes recursos do OpenMP:

* Regiões paralelas
* Loops paralelos
* Redução

In [11]:
%%writefile Pi-V1.0.c
// Versão CPU/OpenMP em FP32.
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    start_time = omp_get_wtime();

    #pragma omp parallel for reduction(+:sum)
    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;
    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing Pi-V1.0.c


In [12]:
%%shell

#clang -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda Pi-V1.0.c -o Pi-V1.0
gcc -fno-lto -fopenmp -fstack-protector Pi-V1.0.c -o Pi-V1.0
./Pi-V1.0

pi = 1.34217727, 100000000 steps, 0.407472 secs


# Exemplo 3: Pi V2.0 na GPU com __target__

A seguir, vamos utilizar a GPU para tentar acelerar nosso programa, utilizando:
* Construção __target__ :

  `#pragma omp target map ...   
  `#pragma omp parallel for
  `     for (i=0;i<N;i++) ...`
   

  Modifique o programa abaixo, introduzindo a melhoria sugerida.
  
  Pode ser necessário fazer uma __redução__ sobre a variável 'sum'

In [13]:
%%writefile pi-par-V2.c
// Primeiro offload para GPU em FP32: target parallel for.
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    start_time = omp_get_wtime();

    #pragma omp target parallel for map(to: step, num_steps) reduction(+:sum)
    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;
    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing pi-par-V2.c


In [14]:
%%shell

#clang -O3 -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda  pi-par-V2.c -o pi-par-V2
gcc -fno-lto -fopenmp  -fstack-protector pi-par-V2.c -o pi-par-V2
./pi-par-V2

pi = 1.34217727, 100000000 steps, 0.850937 secs



__Pergunta:__ O que aconteceu? Compare o tempo de execução com o da CPU.

## Perfilamento de código (profiling)

O NVprof é uma ferramenta de profiling, que fornece diversos contadores de desempenho pelos quais podemos ter uma noção do que pode estar indo mal e limitando o desempenho do programa.

In [15]:
!nvprof ./pi-par-V2

==2090== NVPROF is profiling process 2090, command: ./pi-par-V2
pi = 1.34217727, 100000000 steps, 1.360905 secs
==2090== Profiling application: ./pi-par-V2
==2090== Profiling result:
No kernels were profiled.
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
      API calls:   61.48%  246.84ms         1  246.84ms  246.84ms  246.84ms  cuCtxCreate
                   38.28%  153.68ms         1  153.68ms  153.68ms  153.68ms  cuCtxDestroy
                    0.22%  897.79us        16  56.112us     164ns  891.80us  cuDeviceGetAttribute
                    0.01%  57.680us         2  28.840us  1.3200us  56.360us  cuCtxGetDevice
                    0.00%  13.281us         1  13.281us  13.281us  13.281us  cuDeviceGetName
                    0.00%  6.2390us         2  3.1190us     361ns  5.8780us  cuDeviceGet
                    0.00%  3.9980us         1  3.9980us  3.9980us  3.9980us  cuDeviceGetPCIBusId
                    0.00%  1.9160us         4     479ns     1

# Exemplo 4: Pi V3.0 na GPU, usando __target__ e __teams__

A seguir, vamos utilizar a GPU para tentar acelerar nosso programa, utilizando:
* Construção __target__ :

  `#pragma omp target`    
  `#pragma omp teams`    
  `#pragma omp distribute`    
  `     for (i=0;i<N;i++) ...`


  Modifique o programa abaixo, introduzindo a melhoria sugerida.

  Para este exercício, não utilize a construção  `#pragma omp parallel for`.


In [16]:
%%writefile pi-par-V3.c
// GPU em FP32 usando target teams distribute.
// Esta versão mostra a hierarquia de teams, mas ainda não explora threads/SIMD dentro de cada team.
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    const int teams = 8192;
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    start_time = omp_get_wtime();

    #pragma omp target teams distribute \
        num_teams(teams) map(to: step, num_steps) reduction(+:sum)
    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;
    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing pi-par-V3.c


In [17]:
%%shell

#clang -O3 -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda  pi-par-V3.c -o pi-par-V3
gcc -fno-lto -fopenmp  -fstack-protector pi-par-V3.c -o pi-par-V3
./pi-par-V3

pi = 3.14158964, 100000000 steps, 0.797166 secs


# Exemplo 5: Pi V4 na GPU com a diretiva SIMD

Para obter bom desempenho, é preciso ter paralelismo massivo!

A seguir, vamos utilizar a GPU com as lanes SIMD:
* Construção __target__ :

  `#pragma omp target teams`
  
  `{`
       `#pragma omp distribute parallel for simd`    
  
  `     for (i=0;i<N;i++) ...`


  Modifique o programa abaixo, introduzindo a melhoria sugerida.

  Para este exercício, não utilize a construção  `#pragma omp parallel for`.

In [18]:
%%writefile pi-par-V4.c
// GPU em FP32 usando target teams distribute parallel for simd.
// Esta é a versão recomendada para explorar paralelismo massivo na GPU.
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    const int teams = 8192;
    const int threads_per_team = 256;
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    // Aquecimento: tira criação de contexto CUDA da medição principal.
    //#pragma omp target
    //{ }

    start_time = omp_get_wtime();

    #pragma omp target teams distribute parallel for simd \
        num_teams(teams) thread_limit(threads_per_team) simdlen(32) \
        map(to: step, num_steps) reduction(+:sum)
    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;
    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing pi-par-V4.c


In [19]:
%%shell

#clang -O3 -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda  pi-par-V4.c -o pi-par-V4
gcc -fno-lto -fopenmp  -fstack-protector pi-par-V4.c -o pi-par-V4
./pi-par-V4

pi = 3.14159942, 100000000 steps, 0.518329 secs


### Ajustes de desempenho usados na V4 FP32

A última versão usa `target teams distribute parallel for simd` como uma diretiva combinada. Isso ajuda o compilador a gerar um único kernel com a hierarquia completa: league de `teams`, threads dentro de cada team e lanes SIMD.

Os parâmetros sugeridos são:

- `thread_limit(256)`: valor comum e eficiente para GPUs NVIDIA; cria blocos suficientemente grandes sem exagerar registradores.
- `num_teams(8192)`: cria muitos blocos para manter a GPU ocupada, deixando o runtime escalonar sobre os SMs disponíveis.
- `simdlen(32)`: combina naturalmente com o tamanho de warp NVIDIA.
- aquecimento com uma região `target` vazia: remove a criação do contexto CUDA da medição.

Como agora tudo está em FP32, a precisão será menor. Para aula, isso é útil: fica claro que desempenho e precisão são escolhas de projeto.


In [20]:
!nvprof ./pi-par-V3

==2139== NVPROF is profiling process 2139, command: ./pi-par-V3
pi = 3.14158964, 100000000 steps, 0.510852 secs
==2139== Profiling application: ./pi-par-V3
==2139== Profiling result:
No kernels were profiled.
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
      API calls:   61.26%  151.39ms         1  151.39ms  151.39ms  151.39ms  cuCtxCreate
                   38.42%  94.951ms         1  94.951ms  94.951ms  94.951ms  cuCtxDestroy
                    0.29%  714.37us        16  44.648us     111ns  710.51us  cuDeviceGetAttribute
                    0.02%  49.346us         2  24.673us  1.4290us  47.917us  cuCtxGetDevice
                    0.00%  5.6210us         2  2.8100us     248ns  5.3730us  cuDeviceGet
                    0.00%  5.2820us         1  5.2820us  5.2820us  5.2820us  cuDeviceGetName
                    0.00%  1.4820us         1  1.4820us  1.4820us  1.4820us  cuDeviceGetPCIBusId
                    0.00%  1.3890us         1  1.3890us  1.38


## Observação sobre desempenho: por que a GPU pode perder aqui?

Este exemplo é ótimo para introduzir `target`, `teams`, `distribute` e `simd`, mas ele não é um bom *benchmark* de GPU. O cálculo de π abaixo faz pouco trabalho por elemento, usa `double`, faz divisão em ponto flutuante e termina com uma redução global. Em uma Tesla T4, FP64 é propositalmente fraco quando comparado ao FP32; além disso, o custo de criar contexto CUDA/offload pode aparecer na primeira medição.

Para uma comparação didática mais honesta:

1. aqueça a GPU antes de medir;
2. repita o kernel várias vezes dentro do mesmo processo;
3. separe tempo de inicialização do runtime e tempo de kernel;
4. compare também uma versão `float`, pois GPUs como T4 são muito mais fortes em FP32 do que FP64;
5. mostre que redução é um padrão menos favorável do que operações massivamente paralelas independentes.


In [21]:
%%writefile pi-par-V4-correto.c
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
float step;

int main(void)
{
    const int teams = 8192;
    const int threads_per_team = 256;
    long i;
    float pi, sum = 0.0f;
    double start_time, run_time;

    step = 1.0f / (float) num_steps;

    #pragma omp target
    { }

    start_time = omp_get_wtime();

    #pragma omp target teams distribute parallel for simd \
        num_teams(teams) thread_limit(threads_per_team) simdlen(32) \
        map(to: step, num_steps) reduction(+:sum)
    for (i = 0; i < num_steps; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    pi = step * sum;
    run_time = omp_get_wtime() - start_time;

    printf("pi = %.8f, %ld steps, %.6lf secs\n", pi, num_steps, run_time);
    return 0;
}


Writing pi-par-V4-correto.c


In [22]:
%%shell
gcc -O3 -fno-lto -fopenmp -fstack-protector pi-par-V4-correto.c -o pi-par-V4-correto
./pi-par-V4-correto


pi = 3.14159942, 100000000 steps, 0.360663 secs



### Versão com repetição de kernel

A célula abaixo executa o mesmo kernel várias vezes no mesmo processo. Isso ajuda a mostrar que a primeira chamada costuma incluir custos de inicialização que não fazem parte do cálculo em si.


In [23]:
%%writefile pi-gpu-repeticoes.c
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
static int repeats = 10;

float pi_kernel(long n)
{
    const int teams = 8192;
    const int threads_per_team = 256;
    float step = 1.0f / (float)n;
    float sum = 0.0f;

    #pragma omp target teams distribute parallel for simd \
        num_teams(teams) thread_limit(threads_per_team) simdlen(32) \
        map(to: step, n) reduction(+:sum)
    for (long i = 0; i < n; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    return step * sum;
}

int main(void)
{
    #pragma omp target
    { }

    float pi = 0.0f;
    double t0 = omp_get_wtime();

    for (int r = 0; r < repeats; r++) {
        pi = pi_kernel(num_steps);
    }

    double elapsed = omp_get_wtime() - t0;
    printf("pi = %.8f\n", pi);
    printf("%d repeticoes, %.6lf secs total, %.6lf secs/kernel\n",
           repeats, elapsed, elapsed / repeats);
    return 0;
}


Writing pi-gpu-repeticoes.c


In [24]:
%%shell
gcc -O3 -fno-lto -fopenmp -fstack-protector pi-gpu-repeticoes.c -o pi-gpu-repeticoes
./pi-gpu-repeticoes


pi = 3.14159942
10 repeticoes, 1.692222 secs total, 0.169222 secs/kernel



### Versão `float` para explicar FP32 versus FP64

A Tesla T4 é muito mais rápida em `float` do que em `double`. Esta versão sacrifica precisão, mas evidencia uma diferença arquitetural importante para aulas de GPU.


In [25]:
%%writefile pi-gpu-float.c
#include <stdio.h>
#include <omp.h>

static long num_steps = 100000000;
static int repeats = 10;

float pi_kernel_float(long n)
{
    const int teams = 8192;
    const int threads_per_team = 256;
    float step = 1.0f / (float)n;
    float sum = 0.0f;

    #pragma omp target teams distribute parallel for simd \
        num_teams(teams) thread_limit(threads_per_team) simdlen(32) \
        map(to: step, n) reduction(+:sum)
    for (long i = 0; i < n; i++) {
        float x = ((float)i + 0.5f) * step;
        sum += 4.0f / (1.0f + x * x);
    }

    return step * sum;
}

int main(void)
{
    #pragma omp target
    { }

    float pi = 0.0f;
    double t0 = omp_get_wtime();

    for (int r = 0; r < repeats; r++) {
        pi = pi_kernel_float(num_steps);
    }

    double elapsed = omp_get_wtime() - t0;
    printf("pi = %.8f\n", pi);
    printf("%d repeticoes, %.6lf secs total, %.6lf secs/kernel\n",
           repeats, elapsed, elapsed / repeats);
    return 0;
}


Writing pi-gpu-float.c


In [26]:
%%shell
gcc -O3 -fno-lto -fopenmp -fstack-protector pi-gpu-float.c -o pi-gpu-float
./pi-gpu-float


pi = 3.14159942
10 repeticoes, 1.641366 secs total, 0.164137 secs/kernel



### Um exemplo mais favorável para GPU: SAXPY

Para mostrar aceleração de GPU logo no início, uma operação vetorial grande costuma ser mais didática. Ela expõe paralelismo massivo com pouca sincronização. Depois, o exemplo de π pode ser usado para discutir reduções, precisão dupla e custo de offload.


In [27]:
%%writefile vadd.c

#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#define N 100000
#define TOL  0.0000001

int main(void)
{
    double *a = malloc(sizeof(double) * N);
    double *b = malloc(sizeof(double) * N);
    double *c = malloc(sizeof(double) * N);
    double *res = malloc(sizeof(double) * N);
    int err = 0;

    // Inicialização na CPU: simples e evita mapear o vetor de referência.
    for (int i = 0; i < N; i++) {
        a[i] = (double)i;
        b[i] = 2.0 * (double)i;
        c[i] = 0.0;
        res[i] = a[i] + b[i];
    }

    // Soma vetorial na GPU.
    #pragma omp target teams distribute parallel for simd \
        map(to:a[0:N], b[0:N]) map(from:c[0:N])
    for (int i = 0; i < N; i++) {
        c[i] = a[i] + b[i];
    }

    // Teste na CPU.
    #pragma omp parallel for reduction(+:err)
    for (int i = 0; i < N; i++) {
        double val = c[i] - res[i];
        val = val * val;
        if (val > TOL) err++;
    }

    printf("vectors added with %d errors\n", err);

    free(a);
    free(b);
    free(c);
    free(res);
    return 0;
}


Writing vadd.c


In [28]:
%%shell

#clang -O3 -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda  pi-par-V3.c -o pi-par-V3
gcc -fno-lto -fopenmp  -fstack-protector vadd.c -o vadd
./vadd

vectors added with 0 errors


In [29]:
%%writefile saxpy-openmp-target.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

#define N (1L << 26)
#define REPEATS 20

int main(void)
{
    float *x = malloc(sizeof(float) * N);
    float *y = malloc(sizeof(float) * N);
    float a = 2.0f;

    for (long i = 0; i < N; i++) {
        x[i] = 1.0f;
        y[i] = 2.0f;
    }

    #pragma omp target enter data map(to:x[0:N], y[0:N])

    #pragma omp target
    { }

    double t0 = omp_get_wtime();

    for (int r = 0; r < REPEATS; r++) {
        #pragma omp target teams distribute parallel for simd map(to:a)
        for (long i = 0; i < N; i++) {
            y[i] = a * x[i] + y[i];
        }
    }

    double elapsed = omp_get_wtime() - t0;

    #pragma omp target update from(y[0:10])
    #pragma omp target exit data map(delete:x[0:N], y[0:N])

    printf("y[0] = %.1f, esperado = %.1f\n", y[0], 2.0f + REPEATS * a);
    printf("%d SAXPYs de %ld elementos: %.6lf secs total, %.6lf secs/kernel\n",
           REPEATS, N, elapsed, elapsed / REPEATS);

    free(x);
    free(y);
    return 0;
}


Writing saxpy-openmp-target.c


In [30]:
%%shell
gcc -O3 -fno-lto -fopenmp -fstack-protector saxpy-openmp-target.c -o saxpy-openmp-target
./saxpy-openmp-target


y[0] = 2.0, esperado = 42.0
20 SAXPYs de 67108864 elementos: 0.816652 secs total, 0.040833 secs/kernel
